In [2]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

HDFS_BASE = "hdfs:///user/vagrant"

AIRBNB_CLEANED = f"{HDFS_BASE}/bigdata/processed/airbnb_cleaned"
POLICE_UNIFIED = f"{HDFS_BASE}/bigdata/processed/police_unified"

ANALYTICAL_OUT = f"{HDFS_BASE}/bigdata/analytical/listings_with_police_stats"
POLICE_AGG_OUT = f"{HDFS_BASE}/bigdata/analytical/police_geohash_agg"  # opcjonalne, ale przydatne

print("AIRBNB_CLEANED:", AIRBNB_CLEANED)
print("POLICE_UNIFIED:", POLICE_UNIFIED)
print("ANALYTICAL_OUT:", ANALYTICAL_OUT)
print("POLICE_AGG_OUT:", POLICE_AGG_OUT)

AIRBNB_CLEANED: hdfs:///user/vagrant/bigdata/processed/airbnb_cleaned
POLICE_UNIFIED: hdfs:///user/vagrant/bigdata/processed/police_unified
ANALYTICAL_OUT: hdfs:///user/vagrant/bigdata/analytical/listings_with_police_stats
POLICE_AGG_OUT: hdfs:///user/vagrant/bigdata/analytical/police_geohash_agg


In [3]:
airbnb = spark.read.parquet(AIRBNB_CLEANED)
police = spark.read.parquet(POLICE_UNIFIED)

print("Airbnb rows:", airbnb.count())
airbnb.groupBy("city_code").count().show()

print("Police rows:", police.count())
police.groupBy("city_code", "source").count().show()

airbnb.show(3, truncate=80)
police.show(3, truncate=80)

Airbnb rows: 400


+---------+-----+
|city_code|count|
+---------+-----+
|      LON|  200|
|      NYC|  200|
+---------+-----+

Police rows: 79538


+---------+---------+-----+
|city_code|   source|count|
+---------+---------+-----+
|      LON|UK_POLICE|79138|
|      NYC|     NYPD|  400|
+---------+---------+-----+

+------+---------+----------------------+-----------------+--------------------+--------+---------+---------+---------------------------+------------+--------+----+-----------------+--------------------+----------------+-----------------+------------------+--------------------+
|    id|city_code|neighbourhood_cleansed|       latitude_d|         longitude_d|geohash5|price_num|room_type|              property_type|accommodates|bedrooms|beds|number_of_reviews|review_scores_rating|availability_365|host_is_superhost|host_response_rate|host_acceptance_rate|
+------+---------+----------------------+-----------------+--------------------+--------+---------+---------+---------------------------+------------+--------+----+-----------------+--------------------+----------------+-----------------+------------------+----------------

In [4]:
airbnb.select(
    F.sum(F.col("city_code").isNull().cast("int")).alias("city_nulls"),
    F.sum(F.col("geohash5").isNull().cast("int")).alias("geohash_nulls"),
).show()

police.select(
    F.sum(F.col("city_code").isNull().cast("int")).alias("city_nulls"),
    F.sum(F.col("geohash5").isNull().cast("int")).alias("geohash_nulls"),
).show()

+----------+-------------+
|city_nulls|geohash_nulls|
+----------+-------------+
|         0|            0|
+----------+-------------+

+----------+-------------+
|city_nulls|geohash_nulls|
+----------+-------------+
|         0|            0|
+----------+-------------+



In [ ]:
police_geo_agg = (
    police
    .groupBy("city_code", "geohash5")
    .agg(
        F.count("*").alias("crime_events_total"),
        F.countDistinct("event_id").alias("crime_events_distinct"),
        F.countDistinct("crime_type_id").alias("crime_types_distinct"),
    )
)

print("Police agg rows:", police_geo_agg.count())
police_geo_agg.groupBy("city_code").count().show()
police_geo_agg.show(5, truncate=False)

Police agg rows: 486


+---------+-----+
|city_code|count|
+---------+-----+
|      LON|  435|
|      NYC|   51|
+---------+-----+



+---------+--------+------------------+---------------------+--------------------+
|city_code|geohash5|crime_events_total|crime_events_distinct|crime_types_distinct|
+---------+--------+------------------+---------------------+--------------------+
|LON      |gcjmw   |1                 |1                    |1                   |
|LON      |gcn8t   |1                 |1                    |1                   |
|LON      |gcp7u   |1                 |1                    |1                   |
|LON      |gcqd3   |1                 |1                    |1                   |
|LON      |u10jd   |105               |105                  |11                  |
+---------+--------+------------------+---------------------+--------------------+
only showing top 5 rows



In [ ]:
crime_desc_counts = (
    police
    .filter(F.col("crime_desc").isNotNull())
    .groupBy("city_code", "geohash5", "crime_desc")
    .count()
)

w = (
    F.row_number().over(
        __import__("pyspark").sql.window.Window.partitionBy("city_code","geohash5").orderBy(F.col("count").desc())
    )
)

top_crime_desc = (
    crime_desc_counts
    .withColumn("rn", w)
    .filter(F.col("rn") == 1)
    .select("city_code", "geohash5", F.col("crime_desc").alias("top_crime_desc"), F.col("count").alias("top_crime_desc_count"))
)

police_geo_agg = police_geo_agg.join(top_crime_desc, on=["city_code","geohash5"], how="left")

police_geo_agg.show(5, truncate=False)

+---------+--------+------------------+---------------------+--------------------+---------------------------------+--------------------+
|city_code|geohash5|crime_events_total|crime_events_distinct|crime_types_distinct|top_crime_desc                   |top_crime_desc_count|
+---------+--------+------------------+---------------------+--------------------+---------------------------------+--------------------+
|LON      |gcjmw   |1                 |1                    |1                   |On or near Nanthir Road          |1                   |
|LON      |gcn8t   |1                 |1                    |1                   |On or near Grove Road            |1                   |
|LON      |gcp7u   |1                 |1                    |1                   |On or near Motorway Service Area |1                   |
|LON      |gcqd3   |1                 |1                    |1                   |On or near Cavalier Drive        |1                   |
|LON      |u10jd   |105           

In [ ]:
listings_with_police = (
    airbnb.alias("a")
    .join(
        police_geo_agg.alias("p"),
        on=[F.col("a.city_code") == F.col("p.city_code"), F.col("a.geohash5") == F.col("p.geohash5")],
        how="left"
    )
)

listings_with_police = (
    listings_with_police
    .drop(F.col("p.city_code"))
    .drop(F.col("p.geohash5"))
)

print("Joined rows:", listings_with_police.count())
listings_with_police.select("city_code","geohash5","crime_events_total").show(10, truncate=False)

Joined rows: 400


[Stage 76:================================================>         (5 + 1) / 6]

+---------+--------+------------------+
|city_code|geohash5|crime_events_total|
+---------+--------+------------------+
|NYC      |dr5qg   |1                 |
|NYC      |dr5rk   |8                 |
|NYC      |dr5rk   |8                 |
|NYC      |dr5rk   |8                 |
|NYC      |dr5rk   |8                 |
|NYC      |dr5rk   |8                 |
|NYC      |dr5rk   |8                 |
|NYC      |dr5rk   |8                 |
|NYC      |dr5rk   |8                 |
|NYC      |dr5rk   |8                 |
+---------+--------+------------------+
only showing top 10 rows



In [ ]:
listings_enriched = (
    listings_with_police
    .withColumn("crime_events_total", F.coalesce(F.col("crime_events_total"), F.lit(0)))
    .withColumn("crime_events_distinct", F.coalesce(F.col("crime_events_distinct"), F.lit(0)))
    .withColumn("crime_types_distinct", F.coalesce(F.col("crime_types_distinct"), F.lit(0)))
)

# safety_index: 1 / (1 + crimes_total)
listings_enriched = listings_enriched.withColumn(
    "safety_index",
    (F.lit(1.0) / (F.lit(1.0) + F.col("crime_events_total").cast("double")))
)

listings_enriched = listings_enriched.withColumn(
    "crime_bucket",
    F.when(F.col("crime_events_total") == 0, F.lit("0"))
     .when((F.col("crime_events_total") >= 1) & (F.col("crime_events_total") <= 5), F.lit("1-5"))
     .when((F.col("crime_events_total") >= 6) & (F.col("crime_events_total") <= 20), F.lit("6-20"))
     .otherwise(F.lit("21+"))
)

listings_enriched.select("city_code","crime_events_total","safety_index","crime_bucket").show(10, truncate=False)

[Stage 84:===============>  (5 + 1) / 6][Stage 85:>                 (0 + 1) / 2]

+---------+------------------+------------------+------------+
|city_code|crime_events_total|safety_index      |crime_bucket|
+---------+------------------+------------------+------------+
|NYC      |1                 |0.5               |1-5         |
|NYC      |8                 |0.1111111111111111|6-20        |
|NYC      |8                 |0.1111111111111111|6-20        |
|NYC      |8                 |0.1111111111111111|6-20        |
|NYC      |8                 |0.1111111111111111|6-20        |
|NYC      |8                 |0.1111111111111111|6-20        |
|NYC      |8                 |0.1111111111111111|6-20        |
|NYC      |8                 |0.1111111111111111|6-20        |
|NYC      |8                 |0.1111111111111111|6-20        |
|NYC      |8                 |0.1111111111111111|6-20        |
+---------+------------------+------------------+------------+
only showing top 10 rows



In [ ]:
listings_enriched.groupBy("city_code", "crime_bucket").count().orderBy("city_code","crime_bucket").show()

listings_enriched.groupBy("city_code").agg(
    F.count("*").alias("listings"),
    F.avg("crime_events_total").alias("avg_crimes_in_geohash"),
    F.expr("percentile_approx(crime_events_total, 0.5)").alias("median_crimes_in_geohash"),
    F.avg("safety_index").alias("avg_safety_index"),
).show()

+---------+------------+-----+
|city_code|crime_bucket|count|
+---------+------------+-----+
|      LON|         21+|  200|
|      NYC|         1-5|   30|
|      NYC|         21+|   69|
|      NYC|        6-20|  101|
+---------+------------+-----+



+---------+--------+---------------------+------------------------+--------------------+
|city_code|listings|avg_crimes_in_geohash|median_crimes_in_geohash|    avg_safety_index|
+---------+--------+---------------------+------------------------+--------------------+
|      LON|     200|              1954.31|                    1746|9.308563463314404E-4|
|      NYC|     200|               17.105|                      16| 0.09293492176038617|
+---------+--------+---------------------+------------------------+--------------------+



In [10]:
(police_geo_agg
 .write
 .mode("overwrite")
 .parquet(POLICE_AGG_OUT)
)

(listings_enriched
 .write
 .mode("overwrite")
 .parquet(ANALYTICAL_OUT)
)

print("Saved police agg to :", POLICE_AGG_OUT)
print("Saved final dataset to:", ANALYTICAL_OUT)

26/01/13 02:12:46 WARN hdfs.DFSClient: Caught exception          (30 + 2) / 200]
java.lang.InterruptedException
	at java.lang.Object.wait(Native Method)
	at java.lang.Thread.join(Thread.java:1252)
	at java.lang.Thread.join(Thread.java:1326)
	at org.apache.hadoop.hdfs.DFSOutputStream$DataStreamer.closeResponder(DFSOutputStream.java:716)
	at org.apache.hadoop.hdfs.DFSOutputStream$DataStreamer.endBlock(DFSOutputStream.java:476)
	at org.apache.hadoop.hdfs.DFSOutputStream$DataStreamer.run(DFSOutputStream.java:652)
26/01/13 02:12:52 WARN hdfs.DFSClient: Caught exception          (92 + 2) / 200]
java.lang.InterruptedException
	at java.lang.Object.wait(Native Method)
	at java.lang.Thread.join(Thread.java:1252)
	at java.lang.Thread.join(Thread.java:1326)
	at org.apache.hadoop.hdfs.DFSOutputStream$DataStreamer.closeResponder(DFSOutputStream.java:716)
	at org.apache.hadoop.hdfs.DFSOutputStream$DataStreamer.endBlock(DFSOutputStream.java:476)
	at org.apache.hadoop.hdfs.DFSOutputStream$DataStreamer.

Saved police agg to : hdfs:///user/vagrant/bigdata/analytical/police_geohash_agg
Saved final dataset to: hdfs:///user/vagrant/bigdata/analytical/listings_with_police_stats


In [11]:
final_check = spark.read.parquet(ANALYTICAL_OUT)
print("Final rows:", final_check.count())

final_check.groupBy("city_code").count().show()
final_check.select("city_code","geohash5","crime_events_total","safety_index","crime_bucket").show(5, truncate=False)

Final rows: 400


+---------+-----+
|city_code|count|
+---------+-----+
|      LON|  200|
|      NYC|  200|
+---------+-----+

+---------+--------+------------------+------------------+------------+
|city_code|geohash5|crime_events_total|safety_index      |crime_bucket|
+---------+--------+------------------+------------------+------------+
|NYC      |dr5qg   |1                 |0.5               |1-5         |
|NYC      |dr5rk   |8                 |0.1111111111111111|6-20        |
|NYC      |dr5rk   |8                 |0.1111111111111111|6-20        |
|NYC      |dr5rk   |8                 |0.1111111111111111|6-20        |
|NYC      |dr5rk   |8                 |0.1111111111111111|6-20        |
+---------+--------+------------------+------------------+------------+
only showing top 5 rows

